# Установка зависимостей

In [3]:
!pip install -q pandas pyarrow html2text llama-index-core llama-index-embeddings-huggingface llama-index-retrievers-bm25 llama-index-retrievers-bm25 sentence-transformers
!pip install -q llama-index-llms-openai pymorphy3

In [4]:
import pandas as pd
import html2text
import regex
import numpy as np
from pymorphy3 import MorphAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.postprocessor import SentenceTransformerRerank

# Сплиттинг и обработка чанков

In [5]:
# Стоп-слова для пользовательских запросов
STOP_WORDS = {
    "здравствуйте", "привет", "добрый", "день", "утро", "вечер",
    "подскажите", "помогите", "пожалуйста", "спасибо", "вопрос"
}

# Удаление слишком коротких чанков статей
def remove_short_chunks(nodes, min_words=15):
    filtered = []
    removed = 0

    for node in nodes:
        word_count = len(node.text.split())
        if word_count >= min_words:
            filtered.append(node)
        else:
            removed += 1

    print(f"Удалено коротких чанков: {removed} (меньше {min_words} слов)")
    return filtered

# Очистка пользовательских запросов
def clean_query(query_text: str) -> str:
    if not query_text:
        return ""
    query = query_text.replace("<MONEY>", "цена").replace("<DATE>", "дата")
    query = query.lower()
    query = regex.sub(r'[^\w\s]', ' ', query)
    words = query.split()
    filtered_words = [w for w in words if w not in STOP_WORDS]
    cleaned_query = " ".join(filtered_words)
    return cleaned_query.strip(" .?!")


In [ ]:
print("Загрузка и парсинг статей...")

# Введите свои пути к датасетам (датасет статей, калибровочный датасет и тестовый соответственно)
articles_df = pd.read_feather("/kaggle/input/datasets/vladbad/avito-df/articles.f")
articles_df = articles_df.drop(245, axis=0)

train_semantic_df = pd.read_feather("/kaggle/input/datasets/vladbad/avito-df/calibration.f")
test_df = pd.read_feather("/kaggle/input/datasets/vladbad/avito-df/test.f")

# Настройка парсера
h2t = html2text.HTML2Text()
h2t.ignore_links = True
h2t.ignore_images = True
h2t.body_width = 0

# Создание документов в виде markdown
documents = []
for _, row in articles_df.iterrows():
    md_body = h2t.handle(row['body'])
    full_text = f"# {row['title']}\n\n{md_body}"

    doc = Document(
        text=full_text,
        metadata={"article_id": int(row['article_id'])},
        excluded_embed_metadata_keys=["article_id"],
        excluded_llm_metadata_keys=["article_id"]
    )
    documents.append(doc)

# Разбиваем на чанки по markdown меткам с ограничением на длину чанка
print("Сплиттинг на чанки...")
parser = MarkdownNodeParser(chunk_size=512, chunk_overlap=50)
nodes = parser.get_nodes_from_documents(documents)

# Очищаем фрагменты статей от лишних символов
for node in nodes:
    node.text = regex.sub(r'[\p{Emoji}\u200d]', '', node.text)
    node.text = regex.sub(r'[\*\\]', '', node.text)
    node.text = regex.sub(r'\s+', ' ', node.text)
    node.text = node.text.replace(" .", "")

# Устраняем дублирующиеся чанки (если есть) и отбрасываем слишком короткие
nodes = remove_short_chunks(nodes)
set_nodes = set()
unique_nodes = []
for node in nodes:
  if node.text not in set_nodes:
    set_nodes.add(node.text)
    unique_nodes.append(node)

nodes = unique_nodes

Загрузка и парсинг статей...
Сплиттинг на чанки...
Удалено коротких чанков: 344 (меньше 15 слов)


In [7]:
# Создаем базу запросов пользователей (семантический кэш)
print("Загрузка семантического кэша...")
semantic_documents = []
for _, row in train_semantic_df.iterrows():
    cleaned_query = clean_query(row['query_text'])
    gt = [int(i) for i in row['ground_truth'].split()]
    query_id = row['query_id']

    doc = Document(
        text=cleaned_query,
        metadata={"query_id": int(query_id), "query_text": cleaned_query, "articles": gt},
        excluded_embed_metadata_keys=["article_id"],
        excluded_llm_metadata_keys=["article_id"]
    )
    semantic_documents.append(doc)

Загрузка семантического кэша...


# Индексация, ретривал и алгоритм ранжирования

In [8]:
# Загрузка эмбеддера
print("Инициализация эмбеддингов...")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="intfloat/multilingual-e5-large-instruct",   #intfloat/multilingual-e5-large-instruct
    query_instruction="query: ",
    text_instruction="passage: ",
)

Инициализация эмбеддингов...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [ ]:
# Индексируем документы (делаем отдельные ретриверы для кэша и для базы статей)
print("Индексация...")
vector_index = VectorStoreIndex(nodes)
semantic_index = VectorStoreIndex(semantic_documents)

semantic_retriever = semantic_index.as_retriever(similarity_top_k=4)

vector_retriever = vector_index.as_retriever(similarity_top_k=80)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=80)

# Инициализируем KNN на tfidf
print("TF-IDF + KNN по статьям...")
_knn_aid_text = {}
for doc in documents:
    aid = int(doc.metadata["article_id"])
    _knn_aid_text[aid] = doc.text

knn_article_ids = np.array(sorted(_knn_aid_text.keys()), dtype=np.int64)
knn_article_texts = [_knn_aid_text[int(a)] for a in knn_article_ids]

tfidf_vectorizer = TfidfVectorizer(max_features=100_000, ngram_range=(1, 2), min_df=2)
tfidf_docs = tfidf_vectorizer.fit_transform(knn_article_texts)

knn_tfidf = NearestNeighbors(n_neighbors=min(80, len(knn_article_ids)), metric="cosine")
knn_tfidf.fit(tfidf_docs)


def knn_retrieve_scores(query_text: str, top_k: int = 80) -> dict:
    q_vec = tfidf_vectorizer.transform([query_text])
    k = min(top_k, len(knn_article_ids))
    distances, indices = knn_tfidf.kneighbors(q_vec, n_neighbors=k)
    scores = {}
    for dist, idx in zip(distances[0], indices[0]):
        aid = str(int(knn_article_ids[idx]))
        scores[aid] = 1.0 / (1.0 + float(dist))
    return scores

print(f"KNN создан: {len(knn_article_ids)} статей")

Индексация...
TF-IDF + KNN по статьям...
KNN создан: 792 статей


In [10]:
# Дамми-ключ (не используется, нужен для работы гибридного ретривера)
import os
os.environ["OPENAI_API_KEY"] = "dummy"

In [ ]:
# Гибридный ретривер (BM25 + Dense)
hybrid_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=30,
    num_queries=1,
    mode="reciprocal_rerank",
    use_async=True,
    retriever_weights=[0.8, 0.2]
)

KNN_WEIGHT = 1.0

# Алгоритм ранжирования статей
def get_top_articles(query_text: str) -> str:
    cache_scores = {}
    search_scores = {}
    cache_hits = set()

    # Если схожесть полученного запроса с кэшированным высокая, вытаскиваем релевантную статью (семантический кэш)
    for node in semantic_retriever.retrieve(query_text):
        if node.score > 0.87:
            for aid in node.metadata.get("articles", []):
                aid_str = str(aid)
                cache_scores[aid_str] = max(cache_scores.get(aid_str, 0), node.score * 2.0)
                cache_hits.add(aid_str)

    # Для ранжирования статей берем максимальный скор чанков статьи (поиск по бд статей)
    for node in vector_retriever.retrieve(query_text):
        aid_str = str(node.metadata.get("article_id"))
        score = node.score if node.score is not None else 0.0
        search_scores[aid_str] = max(search_scores.get(aid_str, 0), score)

    # Начисляем итоговые скоры статьям с защитой от дубликатов
    knn_scores = knn_retrieve_scores(query_text, top_k=80)

    final_scores = {}
    all_ids = set(cache_scores) | set(search_scores) | set(knn_scores)
    
    for aid in all_ids:
        score = 0.0
        if aid in cache_scores:
            score += cache_scores[aid]
        if aid in search_scores:
            score += search_scores[aid]
        if aid in knn_scores:
            score += KNN_WEIGHT * knn_scores[aid]

        # Статьи, которые встретились в кэше и в поиске по бд статей одновременно, получают высокий скор
        if aid in cache_hits and aid in search_scores:
            score += 2.0 
            
        final_scores[aid] = score

    sorted_articles = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    top_10_ids = [aid for aid, _ in sorted_articles[:10]]

    return " ".join(top_10_ids)

# Обработка запросов

In [12]:
# Обрабатываем запросы и сохраняем в финальный датафрейм
print("Обработка тестовых запросов...")
test_df['cleaned_query'] = test_df['query_text'].apply(clean_query)

answers = []
for idx, row in test_df.iterrows():
    if idx % 50 == 0:
        print(f"Обработано {idx}/{len(test_df)} запросов...")

    ans = get_top_articles(row['cleaned_query'])
    answers.append(ans)

test_df['answer'] = answers

# Результаты работы смотрите в answers.csv
result_df = test_df[["query_id", "answer"]]
result_df.to_csv("answer.csv", index=False)
print("Готово! Файл answer.csv сохранен.")

Обработка тестовых запросов...
Обработано 0/500 запросов...
Обработано 50/500 запросов...
Обработано 100/500 запросов...
Обработано 150/500 запросов...
Обработано 200/500 запросов...
Обработано 250/500 запросов...
Обработано 300/500 запросов...
Обработано 350/500 запросов...
Обработано 400/500 запросов...
Обработано 450/500 запросов...
Готово! Файл answer.csv сохранен.
